In [ ]:
%%capture
%pip install langchain langchain_openai langchain_community

In [ ]:
import numpy as np
import requests
from typing import Tuple
from langchain_openai import ChatOpenAI
from langchain_core.documents import Document
from langchain_core.tools import Tool
from langchain_core.prompts import PromptTemplate
from langchain_classic.agents import create_openai_tools_agent, AgentExecutor
from langchain_classic import hub
import os; os.environ['OPENAI_API_KEY'] = ### REPLACE WITH API KEY ###

In [ ]:
def find_start_tag(text:str, tag:str, after:int=None, before:int=None) -> int:
    """
    To account for tags having parameters, this function searches for both cases.
    """
    start_1 = text.find(f'<{tag}>', after, before)
    start_2 = text.find(f'<{tag} ', after, before)
    if start_1 == -1:
        return start_2
    elif start_2 == -1:
        return start_1
    return min([start_1, start_2])

def find_tag(text:str, tag:str, after_index:int=0, before_index:int=None) -> Tuple[str, int]:
    """
    Extracts all content within a specified tag.

    Args:
        text (str): The provided text to search through.
        tag (str): The provided tag to search for.
        after_index (int): An index after which the search should begin.

    Returns:
        str: The extracted text.
        int: The index of the end of the end tag.
    """
    # Find initial start and end tags
    start = find_start_tag(text, tag, after_index, before_index)
    end = text.find(f'</{tag}>', start)

    # Break and return if no start tag is found
    if start == -1:
        return '', -1

    # Special treatment for p tags with no closing tag
    if tag == 'p' and '<p>' in text[start:end]:
        end_of_start = text.find('>', start) + 1
        return text[end_of_start:text.find('.', start)+1], text.find('.', start)+1

    # To consider the case where the same tags are embedded within the main one, iteratively search for more
    sub_start = find_start_tag(text, tag, start+1, end)
    while sub_start > -1:
        # Find the next end tag if sub-tags are found
        end = text.find(f'</{tag}>', end+1)
        sub_start = find_start_tag(text, tag, sub_start+1, end)

    end_of_start = text.find('>', start) + 1
    return text[end_of_start:end], end

def strip_tags(text:str, tags:list[str]) -> str:
    """
    This function strips out the start and end tags from some text using a specified list of tags.

    Args:
        text (str): The text to remove tags from.
        tags: (list[str]): A list of tags to be removed.

    Returns:
        str: The stripped text.
    """
    text_stripped = text
    for tag in tags:
        # For each tag, find up to 1000 instances and remove them
        I = 0
        while I < 1000:
            # Finding the start tag
            start_1 = find_start_tag(text_stripped, tag)
            start_2 = text_stripped.find('>', start_1)
            if start_1 >= 0:
                # If start tag found, remove
                text_stripped = text_stripped[:start_1] + (text_stripped[start_2+1:] if start_2 > -1 else '')

            # Finding the end tag
            end_1 = text_stripped.find(f'</{tag}>')
            end_2 = text_stripped.find('>', end_1)
            if end_1 >= 0:
                # If end tag found, remove
                text_stripped = text_stripped[:end_1] + (text_stripped[end_2+1:] if end_2 > -1 else '')

            # If no more tags are being found, then break
            if start_1 < 0 and end_1 < 0:
                break

            I += 1

    return text_stripped

def info_type_evos(name:str) -> str:
    """
    This function can be used to lookup some facts about a specified Pokémon including it's type, Pokédex entries, and it's evolutions.

    Args:
        name (str): The name of the Pokémon to look up.

    Returns:
        str: A concise text summary about the Pokémon.
    """
    # Making HTTPS request
    result = requests.get(f'https://pokemondb.net/pokedex/{name.lower()}')
    html = result.text

    # If successful request, extract information
    if result.ok:
        # The top of the page includes a few useful statements
        paragraphs = []
        I, i = 0, 0
        while I < 5:
            para, i = find_tag(html, 'p', i)
            if para.startswith('<a rel="lightbox"'):
                # Break when you get to the picture
                break
            paragraphs.append(strip_tags(para, ['em', 'a', 'ab', 'q']))
            I += 1

        text_summary = '\n'.join(paragraphs)

        # Gathering key facts from the first table
        facts = strip_tags(find_tag(html, 'table', html.find('<h2>Pokédex data</h2>'))[0], ['tbody', 'tr', 'th', 'td', 'strong', 'a', 'span', 'small'])
        facts = facts.replace('<br>', ' ').replace('<\br>', ' ').replace('\t', '').replace('\n\n\n', '\n\n')

        # Gathering some training facts
        training_facts = strip_tags(find_tag(html, 'table', html.find('<h2>Training</h2>'))[0], ['tbody', 'tr', 'th', 'td', 'a', 'small'])
        training_facts = training_facts.replace('\t', '').replace('\n\n\n', '\n\n')

        # Gathering some breeding facts
        breeding_facts = strip_tags(find_tag(html, 'table', html.find('<h2>Breeding</h2>'))[0], ['tbody', 'tr', 'th', 'td', 'a', 'span', 'small'])
        breeding_facts = breeding_facts.replace('\t', '').replace('\n\n\n', '\n\n')

        # Gathering the evolution information
        evo_info, i = find_tag(html, 'div', html.find('<h2>Evolution'))
        while html.find('infocard-list-evo', i) > -1:
            evo_info_, i = find_tag(html, 'div', i)
            evo_info += evo_info_
        evo_info = strip_tags(evo_info, ['div', 'span', 'a', 'picture', 'source', 'img', 'small', 'i', 'br'])
        evo_info = evo_info.replace(' &middot;', '')

        return text_summary + facts + training_facts + breeding_facts + evo_info
    else:
        raise Exception('Invalid Pokémon name provided')

def stats(name:str) -> str:
    """
    This function can be used to lookup the base stats a specified Pokémon.

    Args:
        name (str): The name of the Pokémon to look up.

    Returns:
        str: A concise text summary of their stats.
    """
    # Making HTTPS request
    result = requests.get(f'https://pokemondb.net/pokedex/{name.lower()}')
    html = result.text

    # If successful request, extract information
    if result.ok:
        # Gathering the base stats
        base_stats = strip_tags(find_tag(html, 'table', html.find('<h2>Base stats</h2>'))[0], ['tbody', 'tr', 'th', 'td', 'div', 'tfoot'])
        base_stats = base_stats.replace('\t', '').replace('\n\n\n', '\n\n').replace('Min', '').replace('Max', '')
        minmax_start = base_stats.find('HP')
        for stat in range(6):
            minmax_start = base_stats.find('\n\n\n', minmax_start)
            base_stats = base_stats[:minmax_start+2] + 'Min Max' + base_stats[minmax_start+2:]

        return base_stats
    else:
        raise Exception('Invalid Pokémon name provided')

def learnset(name:str) -> str:
    """
    This function can be used to lookup the learnset of a specified Pokémon.

    Args:
        name (str): The name of the Pokémon to look up. If you wish to lookup the
                    learnset for a specific generation, then include the generation
                    number directly after the name, separated by a pipe, e.g. "Venonat|3".

    Returns:
        str: A concise text summary of the moves they can learn.
    """
    # Pulling out the specified generation where relevant
    if "|" in name:
        parts = name.split('|')
        name, gen = parts[0], int(parts[1])
    else:
        gen = 9 # Default to the newest generation (currently 9)

    # Making HTTPS request
    result = requests.get(f'https://pokemondb.net/pokedex/{name.lower()}/moves/{gen}')
    html = result.text

    # If successful request, extract information
    if result.ok:

        # String to store all learnset information
        all_moves = ''

        game_st = 0
        G = 0
        while G < 10:
            # Loop through each of the different games in the generation, identified by an ID
            game_st = html.find('#tab-moves-', game_st+10)
            game_ed = html.find('">', game_st)

            G += 1
            if game_st == -1:
                # No more games found
                break

            # Identify the start of the current games information
            moveset_start = html.find(html[game_st+1:game_ed], game_ed)
            game_name = html[html.find('>', game_st)+1:html.find('</a>', game_st)].replace('&#039;', "'")
            game_moves = f'In {game_name}:\n\n'

            for header in ['Moves learnt by level up', 'Egg moves', 'Moves learnt by reminder', 'Moves learnt by TM']:
                # For each type of learnset, find the header to the table
                head = html.find(f'<h3>{header}</h3>', moveset_start)
                if head == -1:
                    continue
                statement = strip_tags(find_tag(html, 'p', head)[0], ['em'])
                if 'does not learn any' in statement or 'cannot be taught' in statement:
                    # If no moves are learnt, continue to the next header
                    continue

                if game_name in ['Legends: Arceus', 'Let\'s Go Pikachu/Let\'s Go Eevee'] and header == 'Egg moves':
                    # Special treatment is needed for Legends: Arceus and Let's Go egg moves
                    continue

                table = strip_tags(find_tag(html, 'table', head)[0], ['thead', 'tr', 'th', 'a', 'div', 'tbody'])
                table = strip_tags(table.replace('</td>', ' '), ['td'])

                start = 0
                while start != -1:
                    # Specific steps to extract the category component from the image
                    start = table.find('title="', start+7)
                    end = table.find('"', start+7)
                    slot = table.find('>', end)+1
                    table = table[:slot] + table[start+7:end] + table[slot:]

                table = strip_tags(table, ['img'])
                redundant_start = table.find('Acc.') + 4
                redundant_end = table.find('Acc.', redundant_start) + 5
                table = table[:redundant_start] + '\n' + table[redundant_end:] # Dealing with some strange formatting

                game_moves += header + '\n' + table + '\n'

            if game_moves != f'In {game_name}:\n\n':
                all_moves += game_moves + '\n'

        if all_moves == '' and gen > 1:
            # If nothing can be found for the current gen, try the generation prior
            return learnset(f'{name}|{gen-1}')
        elif all_moves == '' and gen == 1:
            # Unlikely scenario, but if a Pokémon learns no moves in any gen, they will end up here
            raise Exception('The Pokémon does not learn any moves in the mainline games.')

        return all_moves

    else:
        raise Exception('Invalid Pokémon name or generation specification.')

def move(name:str) -> str:
    """
    This function can be used to lookup details on a specified move.

    Args:
        name (str): The name of the move to lookup.

    Returns:
        str: A concise text summary of details about the move.
    """
    # Making HTTPS request
    result = requests.get(f'https://pokemondb.net/move/{name.lower().replace(' ', '-')}')
    html = result.text

    # If successful request, extract information
    if result.ok:
        # Table with some key facts and figures
        tab = find_tag(html, 'table', html.find('<h2>Move data</h2>'))[0]
        tab = strip_tags(tab, ['tbody', 'th', 'td', 'a', 'img', 'small']).replace('<tr>', '\n')
        tab = strip_tags(tab, ['tr']).replace('&nbsp;', '').replace('\n\t', '\n')

        # Effects description
        desc_start = html.find('>Effects</h2>')
        desc = html[desc_start+13:html.find('<h2', desc_start+13)]
        desc = strip_tags(desc, ['p', 'em', 'a', 'q', 'div', 'b', 'abbr', 'ul', 'li', 'br']).replace('\t', '').\
                replace('\n<h3>Z-Move effects</h3>', '').replace('\n<h3>Changes</h3>', '')

        # Get target information
        targ = find_tag(html, 'p', html.find('<h2>Move target</h2>'))[0]

        return tab.rstrip('\n') + '\n\n' + desc.rstrip('\n') + '\n\n' + targ

def games():
    with open('Games.csv') as file:
        text = file.read()
    return text

def type_adv(types:str):
    with open('Type Data.txt') as file:
        text = file.read()
    off, dfn = types.split('|')
    start = text.find(f'{off.title()} against {dfn.title()}')
    end = text.find('\n', start)
    return text[start:end]

print(type_adv('grass|water'))

Grass against Water = super-effective


In [ ]:
llm = ChatOpenAI(model='gpt-4.1-mini')

tools = [
    Tool(name='PokemonInfoTypeEvos',
        description='Use this tool to retrieve some key facts about a Pokémon specified by name, including their Pokédex entries, type, and evolutions.',
        func=info_type_evos),
    Tool(name='PokemonStats',
        description='Use this tool to retrieve the base stats of a Pokémon specified by name.',
        func=stats),
    Tool(name='PokemonMoves',
        description='Use this tool to retrieve a list of all moves learned by a Pokémon specified by name. To receieve the learnset for a specified generation, suffix the name with "|[generation number]"',
        func=learnset),
    Tool(name='MoveDetails',
        description='Use this tool to retrieve of a move specified by name, including additional effects.',
        func=move)
]

prompt = hub.pull("hwchase17/openai-tools-agent")

agent = create_openai_tools_agent(llm, tools, prompt)

agent_exec = AgentExecutor(agent=agent, tools=tools, verbose=True)

('name', None)
('input_variables', ['agent_scratchpad', 'input'])
('optional_variables', ['chat_history'])
('input_types', {'chat_history': list[typing.Annotated[typing.Union[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')], typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')], typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')], typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')], typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')], typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')], typing.Annotated[langchain_core.messages.ai.AIMessageChunk, Tag(tag='AIMessageChunk')], typing.Annotated[langchain_core.messages.human.HumanMessageChunk, Tag(tag='HumanMessageChunk')], typing.Annotated[langchain_core.messages.chat.ChatMessageChunk, Tag(tag='ChatMessageChunk')], typing.Annotated[langchain_core.messages.system.SystemMessageChunk, T

In [ ]:
agent_exec.invoke({'input':'Does pikachu learn any moves which can cause the opponent to flinch?'})



> Entering new AgentExecutor chain...

Invoking: `PokemonMoves` with `pikachu`


In Scarlet/Violet:

Moves learnt by level up
Lv. Move Type Cat. Power Acc.
1 Charm Fairy Status  —  100  
1 Nasty Plot Dark Status  —  —  
1 Nuzzle Electric Physical  20  100  
1 Play Nice Normal Status  —  —  
1 Quick Attack Normal Physical  40  100  
1 Sweet Kiss Fairy Status  —  75  
1 Tail Whip Normal Status  —  100  
1 Thunder Shock Electric Special  40  100  
4 Thunder Wave Electric Status  —  90  
8 Double Team Normal Status  —  —  
12 Electro Ball Electric Special  —  100  
16 Feint Normal Physical  30  100  
20 Spark Electric Physical  65  100  
24 Agility Psychic Status  —  —  
28 Iron Tail Steel Physical  100  75  
32 Discharge Electric Special  80  100  
36 Thunderbolt Electric Special  90  100  
40 Light Screen Psychic Status  —  —  
44 Thunder Electric Special  110  70  

Egg moves
Move Type Cat. Power Acc.
Charge Electric Special  40  &infin;  
Fake Out Normal Physical  40  100  
Flail Nor

{'input': 'Does pikachu learn any moves which can cause the opponent to flinch?',
 'output': 'Yes, Pikachu can learn the move Fake Out, which has a high priority and causes the opponent to flinch if it hits. However, Fake Out is only successful on the first turn Pikachu is in battle (or after it switches out and back in).'}

In [ ]:
llm.invoke('In generation 7, can chimchar learn fire blast by TM?')

AIMessage(content='In Generation 7 Pokémon games (Sun, Moon, Ultra Sun, Ultra Moon), Chimchar **cannot learn Fire Blast by TM**. Fire Blast is TM38 in Generation 7, but Chimchar and its evolutions do not have Fire Blast as a teachable TM.\n\nChimchar learns strong Fire-type moves primarily through leveling up and breeding or via move tutors (if available), but not TM Fire Blast.\n\nIf you want to teach a strong Fire-type move to Chimchar or its evolutions, consider:\n\n- Flame Wheel (learned by Chimchar by leveling up)\n- Flamethrower (learned by Monferno by leveling up)\n- Fire Punch (can be taught via Move Tutor, depending on the specific game)\n\nBut Fire Blast via TM is not available for Chimchar.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 162, 'prompt_tokens': 21, 'total_tokens': 183, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_

In [ ]:
llm.invoke('What is Metang\'s hidden ability?')

AIMessage(content="Metang's hidden ability is Light Metal. This ability halves the Pokémon's weight, which can affect moves and abilities that depend on weight.", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 28, 'prompt_tokens': 15, 'total_tokens': 43, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4.1-mini-2025-04-14', 'system_fingerprint': 'fp_1fa7bc8436', 'id': 'chatcmpl-DHAOI4ZhxuZjxswmvpb6rwtBx3aav', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019cce13-2d33-75e1-ab4b-c9ab20f223e0-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 15, 'output_tokens': 28, 'total_tokens': 43, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reason

In [ ]:
agent_exec.invoke({'input':'What is Metang\'s weight?'})



> Entering new AgentExecutor chain...

Invoking: `PokeInfo_Scraper` with `Metang`


Metang is a Steel/Psychic type Pokémon introduced in <abbr title="Pokémon Ruby, Sapphire, FireRed, LeafGreen &amp; Emerald">Generation 3</abbr>.

National &#8470;
0375

Type

SteelPsychic

Species
Iron Claw Pokémon

Height
1.2&nbsp;m (3&#8242;11&#8243;)

Weight
202.5&nbsp;kg (446.4&nbsp;lbs)

Abilities
1. Clear Body Light Metal (hidden ability) 

Local &#8470;
0191 (Ruby/Sapphire/Emerald) 0263 (Black 2/White 2) 0200 (Omega Ruby/Alpha Sapphire) 0215 (Sun/Moon &mdash; Alola dex) 0279 (U.Sun/U.Moon &mdash; Alola dex) 0226 (Legends: Z-A) 0130 (The Crown Tundra) 0138 (The Indigo Disk) 



EV yield

2 Defense

Catch rate

3 (0.4% with PokéBall, full HP)


Base Friendship

35 (lower than normal)


Base Exp.
147

Growth Rate
Slow



Egg Groups
Mineral

Gender
Genderless

Egg cycles
40(10,024&ndash;10,280 steps)




HP
60

Min Max
230
324

Attack
75

Min Max
139
273

Defense
100

Min Max
184
328

Sp. Atk
55

M

{'input': "What is Metang's weight?",
 'output': 'Metang weighs 202.5 kg (446.4 lbs).'}

In [ ]:
llm.invoke('What is Metang\'s weight?')

AIMessage(content='Metang weighs approximately 202.8 pounds (92 kilograms).', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 13, 'prompt_tokens': 14, 'total_tokens': 27, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4.1-mini-2025-04-14', 'system_fingerprint': 'fp_1fa7bc8436', 'id': 'chatcmpl-DHAXUnBa9UNFQTFhmDxswxAD4ZgcM', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019cce1b-e1dd-7973-8e9f-e76d8e941faf-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 14, 'output_tokens': 13, 'total_tokens': 27, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})